# Add Single Session to Existing Dataset

This notebook adds a new session to an existing dataset with:
- Windowing (same parameters as existing dataset)
- Even split across train/val
- Updates existing metadata CSV
- Preserves existing data

In [1]:
# Imports
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path
import os
from datetime import datetime

In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Session to add
SESSION_FILE = 'sessions/session_20260316_123521__glucose_alarm__no_background_noise.wav'

# Existing dataset directory
DATASET_DIR = Path('datasets/dataset_w1.5s_h0.25s_20260314')

# Audio parameters (must match existing dataset)
SAMPLE_RATE = 16000
WINDOW_LENGTH = 1.5      # seconds
HOP_LENGTH = 0.25        # seconds

# Derived parameters
WINDOW_SAMPLES = int(WINDOW_LENGTH * SAMPLE_RATE)
HOP_SAMPLES = int(HOP_LENGTH * SAMPLE_RATE)

# Paths
TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'val'
METADATA_FILE = DATASET_DIR / 'dataset_metadata.csv'
BACKUP_METADATA_FILE = DATASET_DIR / f'dataset_metadata_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'

print("Configuration:")
print(f"  Session file: {SESSION_FILE}")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  Window length: {WINDOW_LENGTH}s ({WINDOW_SAMPLES} samples)")
print(f"  Hop length: {HOP_LENGTH}s ({HOP_SAMPLES} samples)")

Configuration:
  Session file: sessions/session_20260316_123521__glucose_alarm__no_background_noise.wav
  Dataset directory: datasets/dataset_w1.5s_h0.25s_20260314
  Window length: 1.5s (24000 samples)
  Hop length: 0.25s (4000 samples)


In [3]:
# Parse session filename
def parse_session_filename(filepath):
    """
    Parse session filename to extract metadata.
    Expected format: session_<timestamp>__<label>__<context>.wav
    """
    filename = Path(filepath).stem
    parts = filename.split('__')
    
    if len(parts) < 3:
        raise ValueError(f"Invalid filename format: {filename}")
    
    session_id = parts[0].replace('session_', '')
    session_label = parts[1]
    context = '__'.join(parts[2:])
    
    return {
        'session_id': session_id,
        'session_label': session_label,
        'context': context,
        'filepath': str(filepath)
    }

# Parse the session
session_metadata = parse_session_filename(SESSION_FILE)

print("Session metadata:")
print(f"  ID: {session_metadata['session_id']}")
print(f"  Label: {session_metadata['session_label']}")
print(f"  Context: {session_metadata['context']}")

Session metadata:
  ID: 20260316_123521
  Label: glucose_alarm
  Context: no_background_noise


In [4]:
# Load existing metadata
print("Loading existing metadata...")
existing_metadata = pd.read_csv(METADATA_FILE)

print(f"\nExisting dataset:")
print(f"  Total windows: {len(existing_metadata)}")
print(f"  Train: {len(existing_metadata[existing_metadata['split'] == 'train'])}")
print(f"  Val: {len(existing_metadata[existing_metadata['split'] == 'val'])}")
print(f"\nClass distribution:")
print(existing_metadata.groupby(['split', 'label']).size())

# Backup existing metadata
existing_metadata.to_csv(BACKUP_METADATA_FILE, index=False)
print(f"\n✓ Backed up metadata to: {BACKUP_METADATA_FILE.name}")

Loading existing metadata...

Existing dataset:
  Total windows: 2860
  Train: 1430
  Val: 1430

Class distribution:
split  label
train  0        1036
       1         394
val    0        1000
       1         430
dtype: int64

✓ Backed up metadata to: dataset_metadata_backup_20260316_125058.csv


In [5]:
# Load and window the session
def slice_audio_into_windows(audio, sample_rate, window_samples, hop_samples):
    """Slice audio into overlapping windows."""
    windows = []
    num_samples = len(audio)
    window_idx = 0
    start_sample = 0
    
    while start_sample + window_samples <= num_samples:
        window_audio = audio[start_sample:start_sample + window_samples]
        start_time = start_sample / sample_rate
        
        windows.append({
            'window_index': window_idx,
            'start_time_seconds': start_time,
            'audio': window_audio
        })
        
        window_idx += 1
        start_sample += hop_samples
    
    return windows

print(f"Loading session: {SESSION_FILE}")
audio, sr = librosa.load(SESSION_FILE, sr=SAMPLE_RATE, mono=True)

duration = len(audio) / sr
print(f"  Duration: {duration:.2f}s")
print(f"  Samples: {len(audio)}")

# Create windows
windows = slice_audio_into_windows(audio, SAMPLE_RATE, WINDOW_SAMPLES, HOP_SAMPLES)
print(f"\n✓ Created {len(windows)} windows")

Loading session: sessions/session_20260316_123521__glucose_alarm__no_background_noise.wav
  Duration: 180.00s
  Samples: 2880000

✓ Created 715 windows


In [6]:
# Split windows evenly between train and val
print("Splitting windows between train and val...")

# Assign label based on session label
if session_metadata['session_label'] == 'glucose_alarm':
    label = 1
elif session_metadata['session_label'] == 'no_glucose_alarm':
    label = 0
else:
    raise ValueError(f"Unknown session label: {session_metadata['session_label']}")

# Split windows evenly: alternating train/val
new_windows = []
for i, window in enumerate(windows):
    # Alternate: even indices to train, odd to val
    split = 'train' if i % 2 == 0 else 'val'
    
    new_windows.append({
        'session_id': session_metadata['session_id'],
        'window_index': window['window_index'],
        'label': label,
        'split': split,
        'start_time_seconds': window['start_time_seconds'],
        'context': session_metadata['context'],
        'audio': window['audio']
    })

# Count split
train_count = sum(1 for w in new_windows if w['split'] == 'train')
val_count = sum(1 for w in new_windows if w['split'] == 'val')

print(f"\nNew windows split:")
print(f"  Train: {train_count} windows")
print(f"  Val: {val_count} windows")
print(f"  Label: {label} ({'alarm' if label == 1 else 'no alarm'})")

Splitting windows between train and val...

New windows split:
  Train: 358 windows
  Val: 357 windows
  Label: 1 (alarm)


In [7]:
# Save new window files
print("\nSaving new window files...")

new_metadata_records = []

for window in new_windows:
    session_id = window['session_id']
    window_index = window['window_index']
    split = window['split']
    audio_data = window['audio']
    
    # Generate filename
    filename = f"{session_id}_window_{window_index:04d}.wav"
    
    # Determine output directory
    if split == 'train':
        output_path = TRAIN_DIR / filename
    else:
        output_path = VAL_DIR / filename
    
    # Save audio
    sf.write(output_path, audio_data, SAMPLE_RATE)
    
    # Record metadata
    new_metadata_records.append({
        'filename': filename,
        'session_id': session_id,
        'window_index': window_index,
        'label': window['label'],
        'split': split,
        'start_time_seconds': window['start_time_seconds'],
        'context': window['context']
    })

print(f"✓ Saved {len(new_metadata_records)} window files")
print(f"  Train directory: {TRAIN_DIR}")
print(f"  Val directory: {VAL_DIR}")


Saving new window files...
✓ Saved 715 window files
  Train directory: datasets/dataset_w1.5s_h0.25s_20260314/train
  Val directory: datasets/dataset_w1.5s_h0.25s_20260314/val


In [9]:
# Combine with existing metadata
print("\nUpdating metadata...")

new_metadata_df = pd.DataFrame(new_metadata_records)
combined_metadata = pd.concat([existing_metadata, new_metadata_df], ignore_index=True)

# Save updated metadata
combined_metadata.to_csv(METADATA_FILE, index=False)

print(f"✓ Updated metadata saved to: {METADATA_FILE}")
print(f"\nUpdated dataset:")
print(f"  Total windows: {len(combined_metadata)}")
print(f"  Train: {len(combined_metadata[combined_metadata['split'] == 'train'])}")
print(f"  Val: {len(combined_metadata[combined_metadata['split'] == 'val'])}")
print(f"\nClass distribution:")
print(combined_metadata.groupby(['split', 'label']).size())


Updating metadata...
✓ Updated metadata saved to: datasets/dataset_w1.5s_h0.25s_20260314/dataset_metadata.csv

Updated dataset:
  Total windows: 3575
  Train: 1788
  Val: 1787

Class distribution:
split  label
train  0        1036
       1         752
val    0        1000
       1         787
dtype: int64


In [10]:
# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print(f"\nAdded session: {session_metadata['session_id']}")
print(f"  Label: {session_metadata['session_label']}")
print(f"  Context: {session_metadata['context']}")
print(f"  Windows created: {len(new_windows)}")
print(f"    Train: {train_count}")
print(f"    Val: {val_count}")

print(f"\nDataset before:")
print(f"  Total: {len(existing_metadata)} windows")
print(f"\nDataset after:")
print(f"  Total: {len(combined_metadata)} windows")
print(f"  Added: {len(new_metadata_records)} windows")

print(f"\nFiles saved to:")
print(f"  {TRAIN_DIR}")
print(f"  {VAL_DIR}")

print(f"\nMetadata:")
print(f"  Updated: {METADATA_FILE}")
print(f"  Backup: {BACKUP_METADATA_FILE}")

print("\n✓ Session successfully added to dataset!")
print("\nNext steps:")
print("  1. Label the new windows using tests/label_training_data.py")
print("  2. Retrain the model with train_cnn_window_split.ipynb")


SUMMARY

Added session: 20260316_123521
  Label: glucose_alarm
  Context: no_background_noise
  Windows created: 715
    Train: 358
    Val: 357

Dataset before:
  Total: 2860 windows

Dataset after:
  Total: 3575 windows
  Added: 715 windows

Files saved to:
  datasets/dataset_w1.5s_h0.25s_20260314/train
  datasets/dataset_w1.5s_h0.25s_20260314/val

Metadata:
  Updated: datasets/dataset_w1.5s_h0.25s_20260314/dataset_metadata.csv
  Backup: datasets/dataset_w1.5s_h0.25s_20260314/dataset_metadata_backup_20260316_125058.csv

✓ Session successfully added to dataset!

Next steps:
  1. Label the new windows using tests/label_training_data.py
  2. Retrain the model with train_cnn_window_split.ipynb


In [11]:
# Display sample of new windows
print("\nSample of new windows:")
new_metadata_df.head(20)


Sample of new windows:


,filename,session_id,window_index,label,split,start_time_seconds,context
0,20260316_123521_window_0000.wav,20260316_123521,0,1,train,0.00,no_background_noise
1,20260316_123521_window_0001.wav,20260316_123521,1,1,val,0.25,no_background_noise
2,20260316_123521_window_0002.wav,20260316_123521,2,1,train,0.50,no_background_noise
3,20260316_123521_window_0003.wav,20260316_123521,3,1,val,0.75,no_background_noise
4,20260316_123521_window_0004.wav,20260316_123521,4,1,train,1.00,no_background_noise
5,20260316_123521_window_0005.wav,20260316_123521,5,1,val,1.25,no_background_noise
6,20260316_123521_window_0006.wav,20260316_123521,6,1,train,1.50,no_background_noise
7,20260316_123521_window_0007.wav,20260316_123521,7,1,val,1.75,no_background_noise
8,20260316_123521_window_0008.wav,20260316_123521,8,1,train,2.00,no_background_noise
9,20260316_123521_window_0009.wav,20260316_123521,9,1,val,2.25,no_background_noise
